# Xcapit FHE-ML Quickstart

This notebook provides a quick introduction to the Xcapit FHE-ML SDK for
privacy-preserving machine learning using Fully Homomorphic Encryption.

## What You'll Learn
1. Setting up the FHE encryption context
2. Encrypting data with CKKS scheme
3. Training models on encrypted data
4. Making predictions while data remains encrypted

## Prerequisites
```bash
pip install xcapit-fhe-ml
```

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# SDK imports
import sys
sys.path.insert(0, '..')

from sdk import (
    # Encryption
    FHEContextManager,
    CKKSEncryptor,
    SecurityLevel,
    # Data utilities
    SecureDataLoader,
    # Models
    FHEModel,
    LinearRegression,
    ModelConfig,
)

print("SDK imported successfully!")

## Step 1: Generate Sample Data

We'll create a simple regression dataset to demonstrate the workflow.

In [ ]:
# Generate synthetic regression data
X, y = make_regression(
    n_samples=200,
    n_features=5,
    noise=10,
    random_state=42
)

# Normalize features (important for FHE)
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X = scaler_X.fit_transform(X)
y = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Features: {X.shape[1]}")
print(f"\nFeature ranges: [{X.min():.2f}, {X.max():.2f}]")
print(f"Target range: [{y.min():.2f}, {y.max():.2f}]")

## Step 2: Setup FHE Encryption Context

The FHE context defines the encryption parameters including:
- **Polynomial modulus degree**: Controls precision vs performance
- **Security level**: 128, 192, or 256 bits
- **Scale**: Precision for floating-point operations

In [ ]:
# Create FHE context manager
context_manager = FHEContextManager()

# Generate context with 128-bit security
context_manager.generate_context(
    poly_modulus_degree=8192,  # 8192 is good balance of security/performance
    security_level=SecurityLevel.TC128,
)

print("FHE Context created!")
print(f"  Polynomial degree: 8192")
print(f"  Security level: 128-bit")
print(f"  Slots available: {context_manager.context.n_slots}")

In [ ]:
# Create encryptor
encryptor = CKKSEncryptor(context_manager)

print("Encryptor ready!")

## Step 3: Encrypt the Data

We use `SecureDataLoader` to encrypt our training and test data.

In [ ]:
# Create data loader
loader = SecureDataLoader(encryptor)

# Encrypt training data
print("Encrypting training data...")
encrypted_train = loader.encrypt_dataset(X_train, y_train)
print(f"  Encrypted {len(encrypted_train.X)} training samples")

# Encrypt test data (features only - we keep y_test for evaluation)
print("\nEncrypting test data...")
encrypted_test = loader.encrypt_matrix(X_test)
print(f"  Encrypted {len(encrypted_test)} test samples")

## Step 4: Train Model on Encrypted Data

Now we train a Linear Regression model directly on the encrypted data.
The model never sees the plaintext values!

In [ ]:
# Configure training
config = ModelConfig(
    learning_rate=0.1,
    n_epochs=50,
    verbose=True,  # Show training progress
)

# Create model
model = LinearRegression(config=config, encryptor=encryptor)

print("Training on encrypted data...")
print("(This may take a moment due to FHE operations)\n")

In [ ]:
# Train the model
model.fit(encrypted_train)

print(f"\nTraining complete!")
print(f"Final loss: {model.history.losses[-1]:.6f}")
print(f"Model state: {model.state.value}")

## Step 5: Make Predictions on Encrypted Data

Predictions are also computed on encrypted data.

In [ ]:
# Predict on encrypted test data
print("Making predictions on encrypted test data...")
encrypted_predictions = model.predict(encrypted_test)

print(f"Got {len(encrypted_predictions)} encrypted predictions")

In [ ]:
# Decrypt predictions to evaluate
y_pred = []
for enc_pred in encrypted_predictions:
    decrypted = encryptor.decrypt_vector(enc_pred)
    y_pred.append(decrypted[0])  # First element contains the prediction

y_pred = np.array(y_pred)

# Calculate metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nResults:")
print(f"  MSE: {mse:.4f}")
print(f"  R² Score: {r2:.4f}")

In [ ]:
# Show sample predictions
print("\nSample predictions vs actual:")
print(f"{'Actual':>10} {'Predicted':>10} {'Error':>10}")
print("-" * 32)
for i in range(min(10, len(y_test))):
    error = abs(y_test[i] - y_pred[i])
    print(f"{y_test[i]:>10.3f} {y_pred[i]:>10.3f} {error:>10.3f}")

## Step 6: Model Inspection

After training, we can inspect the learned parameters.

In [ ]:
# Get model parameters
params = model.get_params()

print("Model Parameters:")
print(f"  Weights: {params['weights']}")
print(f"  Bias: {params['bias']:.4f}")
print(f"  Features: {params['n_features']}")
print(f"  State: {params['state']}")

## Quick Alternative: Plaintext Training

For faster iteration during development, you can train on plaintext
and then use the model for encrypted predictions.

In [ ]:
# Train on plaintext (much faster)
fast_model = LinearRegression(config=ModelConfig(n_epochs=100, learning_rate=0.1))
fast_model._fit_plaintext(X_train, y_train)

# Predict on plaintext for validation
y_pred_fast = fast_model._predict_plaintext(X_test)

print(f"Plaintext training results:")
print(f"  R² Score: {r2_score(y_test, y_pred_fast):.4f}")
print(f"\nThis model can still make predictions on encrypted data!")

## Summary

In this quickstart, you learned:

1. **Setup**: Create FHE context with `FHEContextManager`
2. **Encrypt**: Use `SecureDataLoader` to encrypt data
3. **Train**: Fit models on encrypted data
4. **Predict**: Make predictions while data stays encrypted

### Next Steps
- Try other models: `LogisticRegression`, `DecisionTree`, `KMeans`
- Explore the CLI: `xcapit-fhe --help`
- Check blockchain integration for model verification

### Key Points
- Always normalize data before encryption (range [-1, 1] or similar)
- FHE operations are slower than plaintext - start with small datasets
- Use plaintext training for development, encrypted for production